# ANÁLISIS COMPLETO WAGE11
## Limpieza, Preprocesamiento y Modelado Predictivo de Salarios

**Dataset:** Wage11.xlsx | **Variables:** 15 | **Observaciones:** 935

## SECCIÓN 1: IMPORTAR LIBRERÍAS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
print('Librerias importadas correctamente')

## SECCIÓN 2: CARGAR DATOS

In [ ]:
# Cargar archivo Excel
df = pd.read_excel('Wage11.xlsx')
df_original = df.copy()
print(f'Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas')
print(f'Columnas: {list(df.columns)}')
df.head()

## SECCIÓN 3: EXPLORACIÓN INICIAL

In [ ]:
print('Estadísticas descriptivas:')
print(df.describe().round(2))
print('\nTipos de datos:')
print(df.dtypes)

## SECCIÓN 4: ELIMINAR DUPLICADOS

**Justificación:** Los duplicados distorsionan el análisis y el modelo. Se mantiene el primer registro de cada grupo.

In [ ]:
# Detectar duplicados (excluyendo id)
num_dup = df.duplicated(subset=df.columns.difference(['id'])).sum()
print(f'Duplicados encontrados: {num_dup} ({num_dup/len(df)*100:.2f}%)')

df = df.drop_duplicates(subset=df.columns.difference(['id']), keep='first').reset_index(drop=True)
print(f'Registros originales: {len(df_original)}')
print(f'Registros después limpieza: {len(df)}')
print(f'Eliminados: {len(df_original)-len(df)}')

## SECCIÓN 5: ANÁLISIS DE VALORES FALTANTES

In [ ]:
faltantes = pd.DataFrame({
    'Variable': df.columns,
    'Faltantes': df.isnull().sum().values,
    'Porcentaje': (df.isnull().sum().values / len(df) * 100).round(2)
})
faltantes_solo = faltantes[faltantes['Faltantes'] > 0].sort_values('Faltantes', ascending=False)
if len(faltantes_solo) > 0:
    print('Variables con valores faltantes:')
    print(faltantes_solo.to_string(index=False))
else:
    print('No hay valores faltantes')
print(f'\nTotal faltantes: {df.isnull().sum().sum()}')

## SECCIÓN 6: IMPUTACIÓN DE VALORES FALTANTES

- **MEDIANA** para wage, educ, exper, black, sibs (robusta ante outliers)
- **MEDIA** para IQ, KWW (distribuciones simétricas)

In [ ]:
# Imputación con MEDIANA
vars_mediana = ['wage', 'educ', 'exper', 'black', 'sibs']
for var in vars_mediana:
    if df[var].isnull().sum() > 0:
        df[var].fillna(df[var].median(), inplace=True)

# Imputación con MEDIA
vars_media = ['IQ', 'KWW']
for var in vars_media:
    if df[var].isnull().sum() > 0:
        df[var].fillna(df[var].mean(), inplace=True)

print(f'Faltantes después imputación: {df.isnull().sum().sum()}')

## SECCIÓN 7: NORMALIZACIÓN Y ESTANDARIZACIÓN

- **MinMaxScaler [0,1]** para WAGE
- **StandardScaler (z-score)** para KWW

In [ ]:
# Normalización WAGE
scaler_wage = MinMaxScaler()
df['wage_normalizado'] = scaler_wage.fit_transform(df[['wage']])
print(f'WAGE normalizado - min: {df["wage_normalizado"].min():.4f}, max: {df["wage_normalizado"].max():.4f}')

# Estandarización KWW
scaler_kww = StandardScaler()
df['KWW_estandarizado'] = scaler_kww.fit_transform(df[['KWW']])
print(f'KWW estandarizado - media: {df["KWW_estandarizado"].mean():.6f}, std: {df["KWW_estandarizado"].std():.4f}')

## SECCIÓN 8: INGENIERÍA DE CARACTERÍSTICAS

In [ ]:
# salary_per_hour: productividad horaria
df['salary_per_hour'] = df['wage'] / df['hours'].replace(0, np.nan)
print(f'salary_per_hour promedio: {df["salary_per_hour"].mean():.2f}')

# educ_exper_ratio: balance educacion vs experiencia
df['educ_exper_ratio'] = df['educ'] / (df['exper'] + 1)
print(f'educ_exper_ratio promedio: {df["educ_exper_ratio"].mean():.2f}')

## SECCIÓN 9: ANÁLISIS DE CORRELACIONES

In [ ]:
vars_interes = ['wage','educ','exper','IQ','KWW','age','hours','tenure','salary_per_hour','educ_exper_ratio','married','black']
corr = df[vars_interes].corr()['wage'].sort_values(ascending=False)
print('Correlación con WAGE:')
for v, c in corr.items():
    if v != 'wage':
        bar = '#' * int(abs(c)*20)
        print(f'  {v:<25} {c:>8.4f}  {bar}')

## SECCIÓN 10: PREPARACIÓN Y DIVISIÓN DE DATOS (80/20)

In [ ]:
X = df[['educ','exper','hours','IQ','KWW','tenure','age','married','black','south','urban','salary_per_hour','educ_exper_ratio']].copy()
y = df['wage'].copy()

# Eliminar NaNs residuales
idx = X.notna().all(axis=1) & y.notna()
X = X[idx].reset_index(drop=True)
y = y[idx].reset_index(drop=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Muestras totales: {len(X)}')
print(f'Entrenamiento (80%): {len(X_train)}')
print(f'Prueba (20%): {len(X_test)}')

## SECCIÓN 11: ENTRENAR MODELO DE REGRESIÓN LINEAL

In [ ]:
modelo = LinearRegression()
modelo.fit(X_train, y_train)

coefs = pd.DataFrame({'Variable': X.columns, 'Coeficiente': modelo.coef_}).sort_values('Coeficiente', ascending=False)
print(f'Intercepto: {modelo.intercept_:.4f}')
print('\nCoeficientes del modelo:')
print(coefs.to_string(index=False))

## SECCIÓN 12: EVALUACIÓN DEL MODELO

In [ ]:
y_train_pred = modelo.predict(X_train)
y_test_pred  = modelo.predict(X_test)

r2_train  = r2_score(y_train, y_train_pred)
r2_test   = r2_score(y_test, y_test_pred)
mae_test  = mean_absolute_error(y_test, y_test_pred)
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))

print('='*60)
print(f'  R2  Entrenamiento : {r2_train:.4f} ({r2_train*100:.2f}%)')
print(f'  R2  Prueba        : {r2_test:.4f}  ({r2_test*100:.2f}%)')
print(f'  MAE Prueba        : {mae_test:.2f}')
print(f'  RMSE Prueba       : {rmse_test:.2f}')
print(f'  Overfitting       : {r2_train-r2_test:.4f}')
print('='*60)

## SECCIÓN 13: CONCLUSIONES FINALES

In [ ]:
print('='*70)
print('CONCLUSIONES FINALES')
print('='*70)
print(f'Registros originales      : {len(df_original)}')
print(f'Registros limpios         : {len(df)}')
print(f'Duplicados eliminados     : {len(df_original)-len(df)}')
print(f'Nuevas features           : salary_per_hour, educ_exper_ratio')
print(f'Transformaciones          : WAGE normalizado, KWW estandarizado')
print(f'Variables en modelo       : {len(X.columns)}')
print(f'R2 Test                   : {r2_test*100:.2f}%')
print(f'MAE                       : {mae_test:.2f}')
print('='*70)

## SECCIÓN 14: GUARDAR ARCHIVO PROCESADO

In [ ]:
df.to_excel('Wage11_PROCESADO.xlsx', index=False)
print('Archivo guardado: Wage11_PROCESADO.xlsx')
print(f'Registros: {len(df)} | Columnas: {len(df.columns)}')